# Test Registered Tapis MODFLOW Apps

This notebook logs into the `portals` tenant, inspects the registered apps, submits test jobs against the `ptdatax.project.PTDATAX-272` storage system, polls job status, and lists archived outputs.

Notes:
- `modflow-usg` uses the Carrizo-Wilcox USG model.
- `modflow-2000` uses the Yegua-Jackson MODFLOW-2000 model.
- `modflow-96` uses the Trinity Hill Country MODFLOW-96 model.
- `modflow6` uses the TWDB Gulf Coast Aquifer System northern portion GAM v4.1 files through generic individual Tapis inputs.

In [1]:
import json
import os
import time
from datetime import datetime
from getpass import getpass
from pathlib import Path
from pprint import pprint

import requests

requests.packages.urllib3.disable_warnings()

In [2]:
# Tenant and system configuration.
BASE_URL = "https://portals.tapis.io"
SYSTEM_ID = "ptdatax.project.PTDATAX-272"
MODEL_ROOT = "workingGAMs"
commit_version = 'f785424'
EXPECTED_APP_VERSION = f"0.0.{commit_version}"
EXPECTED_IMAGE_TAG = f"sha-{commit_version}"
EXPECTED_ALLOCATION = "PT2050-DataX"
EXPECTED_PORTAL_TAG = "portalName: ALL"
ALLOCATION_SCHEDULER_ARG = f"-A {EXPECTED_ALLOCATION}"
MF6_GULF_COAST_DIR = "twdb_gam_collection/Gulf_Coast_Aquifer_System_northern_portion_GAM_version_4.1/glfc_n_v4_tacc/gma14"

# Known registered app ids.
APP_CASES = {
    "modflow-usg-simulation": {
        "enabled": True,
        "image_name": "modflow-usg",
        "description": "Carrizo-Wilcox MODFLOW-USG generated from package inputs",
        "file_inputs": {
            "mfusg-bas": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.bas",
            "mfusg-dis": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.dis",
            "mfusg-drn": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.drn",
            "mfusg-evt": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.evt",
            "mfusg-ghb": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.ghb",
            "mfusg-gnc": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.gnc",
            "mfusg-hfb": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.hfb",
            "mfusg-lpf": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.mod.lpf",
            "mfusg-oc": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.oc",
            "mfusg-rch": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.rch",
            "mfusg-riv": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.riv",
            "mfusg-sms": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.sms",
            "mfusg-wel": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.wel",
            "mfusg-support-01": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/evt.depth.ref",
            "mfusg-support-02": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/evt.nodes.ref",
            "mfusg-support-03": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/evt.rate.ref",
            "mfusg-support-04": f"{MODEL_ROOT}/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/evt.top.ref",
        },
        "expect": "FINISHED",
    },
    # "modflow-2000-simulation": {
    #     "enabled": True,
    #     "image_name": "modflow-2000",
    #     "description": "Yegua-Jackson MODFLOW-2000 generated from package inputs",
    #     "file_inputs": {
    #         "mf2000-bas": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.bas",
    #         "mf2000-bcf": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.bcf",
    #         "mf2000-dis": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.dis",
    #         "mf2000-drn": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.drn",
    #         "mf2000-evt": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.evt",
    #         "mf2000-ghb": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.ghb",
    #         "mf2000-gmg": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.gmg",
    #         "mf2000-oc": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.oc",
    #         "mf2000-rch": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.rch",
    #         "mf2000-res": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.res",
    #         "mf2000-str": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.str",
    #         "mf2000-wel": f"{MODEL_ROOT}/Yequa_Jackson/Yegua_Jackson_Model_Only/CD-2_ygjk_model/Modflow_2000/ygjk_tr.wel",
    #     },
    #     "expect": "FINISHED",
    # },
    "modflow-96-simulation": {
        "enabled": True,
        "image_name": "modflow-96",
        "description": "Trinity Hill Country MODFLOW-96 generated from package inputs",
        "file_inputs": {
            "mf96-bas": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/bas.dat",
            "mf96-bcf": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/bcf.dat",
            "mf96-discret": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/discret.dat",
            "mf96-drn": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/drn.dat",
            "mf96-ghb": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/ghb.dat",
            "mf96-oc": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/oc.dat",
            "mf96-output": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/output.dat",
            "mf96-rch": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/rch.dat",
            "mf96-riv": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/riv.dat",
            "mf96-sor": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/sor.dat",
            "mf96-wel": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/wel.dat",
            "mf96-budget": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/budget.dat",
            "mf96-heads": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/heads.dat",
            "mf96-ddown": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/ddown.dat",
            "mf96-mt3d-flo": f"{MODEL_ROOT}/Trinity_hill_country/Trinity_hill_country_model_only/modfl_96/ststate/mt3d.flo",
        },
        "expect": "FINISHED",
    },
    "modflow6-simulation": {
        "enabled": True,
        "image_name": "modflow6",
        "description": "Gulf Coast northern v4.1 MODFLOW 6 GAM with generated name files",
        "file_inputs": {
            "mf6-support-01": f"{MF6_GULF_COAST_DIR}/array_data",
            "mf6-tdis": f"{MF6_GULF_COAST_DIR}/gma14.tdis",
            "mf6-ims": f"{MF6_GULF_COAST_DIR}/gma14.ims",
            "mf6-dis": f"{MF6_GULF_COAST_DIR}/gma14.dis",
            "mf6-ic": f"{MF6_GULF_COAST_DIR}/gma14.ic",
            "mf6-npf": f"{MF6_GULF_COAST_DIR}/gma14.npf",
            "mf6-sto": f"{MF6_GULF_COAST_DIR}/gma14.sto",
            "mf6-oc": f"{MF6_GULF_COAST_DIR}/gma14.oc",
            "mf6-wel": f"{MF6_GULF_COAST_DIR}/gma14.wel",
            "mf6-rcha": f"{MF6_GULF_COAST_DIR}/gma14_rch_sc.rcha",
            "mf6-rcha-02": f"{MF6_GULF_COAST_DIR}/gma14_rch_oc.rcha",
            "mf6-drn": f"{MF6_GULF_COAST_DIR}/gma14.drn",
            "mf6-riv": f"{MF6_GULF_COAST_DIR}/gma14.riv",
            "mf6-ghb": f"{MF6_GULF_COAST_DIR}/gma14.ghb",
            "mf6-csub": f"{MF6_GULF_COAST_DIR}/gma14.csub",
            "mf6-obs": f"{MF6_GULF_COAST_DIR}/gma14.obs",
            "mf6-support-02": f"{MF6_GULF_COAST_DIR}/gma14.irr",
            "mf6-support-03": f"{MF6_GULF_COAST_DIR}/gma14.csub.obs",
        },
        "expect": "FINISHED",
    },
}


In [3]:
def tapis_url(path: str) -> str:
    path = path.lstrip("/")
    return f"tapis://{SYSTEM_ID}/{path}"


def get_token(username: str, password: str) -> str:
    response = requests.post(
        f"{BASE_URL}/v3/oauth2/tokens",
        json={"username": username, "password": password, "grant_type": "password"},
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    return payload["result"]["access_token"]["access_token"]


def api_headers(token: str) -> dict:
    return {
        "X-Tapis-Token": token,
        "Content-Type": "application/json",
    }


def api_get(token: str, path: str, **params):
    response = requests.get(
        f"{BASE_URL}{path}",
        headers=api_headers(token),
        params=params or None,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["result"]


def api_post(token: str, path: str, body: dict):
    response = requests.post(
        f"{BASE_URL}{path}",
        headers=api_headers(token),
        data=json.dumps(body),
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["result"]


In [4]:
username = os.environ.get("TAPIS_USERNAME", "").strip() or input("Tapis username: ").strip()
password = os.environ.get("TAPIS_PASSWORD", "") or getpass("Tapis password: ")
token = get_token(username, password)
print(f"Authenticated as {username}")

Tapis username:  wmobley
Tapis password:  ········


Authenticated as wmobley


In [5]:
system_info = api_get(token, f"/v3/systems/{SYSTEM_ID}")
print("System:", system_info["id"])
print("Root dir:", system_info["rootDir"])
print("Shared with:", system_info.get("sharedWithUsers", []))

System: ptdatax.project.PTDATAX-272
Root dir: /corral-repl/tacc/aci/PT2050/projects/PTDATAX-272
Shared with: ['wma_prtl', 'wmobley', 'sawp33', 'lpearson']


In [6]:
def get_app(app_id: str) -> dict:
    return api_get(token, f"/v3/apps/{app_id}")


def get_named_file_inputs(app_def: dict) -> dict:
    job_attrs = app_def.get("jobAttributes", {})
    return {item["name"]: item for item in job_attrs.get("fileInputs", [])}


def scheduler_option_args(app_def: dict) -> list[str]:
    parameter_set = app_def.get("jobAttributes", {}).get("parameterSet", {})
    return [item.get("arg", "") for item in parameter_set.get("schedulerOptions", [])]


def build_job_file_inputs(app_def: dict, source_map: dict) -> list[dict]:
    named_inputs = get_named_file_inputs(app_def)
    file_inputs = []
    for name, rel_path in source_map.items():
        if name not in named_inputs:
            raise KeyError(f"Input {name!r} is not defined on app {app_def['id']}")
        file_inputs.append(
            {
                "name": name,
                "sourceUrl": tapis_url(rel_path),
            }
        )
    return file_inputs


def make_job_body(app_def: dict, case: dict) -> dict:
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    body = {
        "name": f"test-{app_def['id']}-{timestamp}",
        "appId": app_def["id"],
        "appVersion": app_def["version"],
    }
    file_inputs = build_job_file_inputs(app_def, case.get("file_inputs", {}))
    if file_inputs:
        body["fileInputs"] = file_inputs
    if ALLOCATION_SCHEDULER_ARG not in scheduler_option_args(app_def):
        body["parameterSet"] = {
            "schedulerOptions": [
                {
                    "name": "TACC Allocation",
                    "arg": ALLOCATION_SCHEDULER_ARG,
                }
            ]
        }
    return body


def submit_job(body: dict) -> dict:
    return api_post(token, "/v3/jobs/submit", body)


def get_job(job_uuid: str) -> dict:
    return api_get(token, f"/v3/jobs/{job_uuid}")


def get_job_status(job_uuid: str) -> dict:
    return api_get(token, f"/v3/jobs/{job_uuid}/status")


def api_get_optional(token: str, path: str, **params):
    response = requests.get(
        f"{BASE_URL}{path}",
        headers=api_headers(token),
        params=params or None,
        timeout=60,
    )
    if response.status_code >= 400:
        return {"ok": False, "status_code": response.status_code, "text": response.text}
    payload = response.json()
    return {"ok": True, "result": payload.get("result")}


def get_job_history(job_uuid: str) -> dict:
    candidate_paths = [
        f"/v3/jobs/{job_uuid}/history",
        f"/v3/jobs/{job_uuid}/events",
        f"/v3/jobs/{job_uuid}/messages",
    ]
    attempts = []
    for path in candidate_paths:
        result = api_get_optional(token, path)
        attempts.append({"path": path, **result})
        if result.get("ok"):
            return {"path": path, "result": result.get("result"), "attempts": attempts}
    return {"path": None, "result": None, "attempts": attempts}


def list_outputs(job_uuid: str, output_path: str = "") -> list:
    suffix = f"/{output_path.lstrip('/')}" if output_path else "/"
    return api_get(token, f"/v3/jobs/{job_uuid}/output/list{suffix}")


def summarize_failure_value(value):
    if value in (None, "", [], {}):
        return []
    if isinstance(value, dict):
        lines = []
        for key in ("status", "state", "phase", "message", "errorMessage", "lastMessage", "created", "ended", "remoteOutcome"):
            if value.get(key):
                lines.append(f"{key}: {value[key]}")
        return lines
    if isinstance(value, list):
        lines = []
        for item in value[-8:]:
            if isinstance(item, dict):
                message = item.get("message") or item.get("description") or item.get("event") or item.get("status") or item.get("state")
                timestamp = item.get("created") or item.get("createdAt") or item.get("timestamp") or item.get("time")
                if message and timestamp:
                    lines.append(f"{timestamp}: {message}")
                elif message:
                    lines.append(str(message))
                else:
                    lines.append(json.dumps(item, sort_keys=True))
            else:
                lines.append(str(item))
        return lines
    return [str(value)]


def print_job_diagnostics(job_uuid: str) -> None:
    print("" + "=" * 100)
    print(f"Failure diagnostics for job: {job_uuid}")

    sections = []
    for label, fetcher in (
        ("Job status", lambda: get_job_status(job_uuid)),
        ("Job detail", lambda: get_job(job_uuid)),
        ("History/messages", lambda: get_job_history(job_uuid)),
    ):
        try:
            value = fetcher()
        except Exception as exc:
            sections.append((label, [f"Unable to query {label.lower()}: {exc}"]))
            continue

        if label == "History/messages":
            lines = [f"endpoint: {value['path']}"]
            lines.extend(summarize_failure_value(value.get("result")) or summarize_failure_value(value.get("attempts")))
        else:
            lines = summarize_failure_value(value)
        sections.append((label, lines or ["No diagnostic message returned."]))

    for label, lines in sections:
        print(f"{label}:")
        for line in lines:
            print(f"  {line}")

    print("Archived outputs:")
    try:
        outputs = list_outputs(job_uuid)
        if not outputs:
            print("  No archived outputs listed.")
        for item in outputs:
            print(f"  - {item.get('path', item.get('name'))}")
    except Exception as exc:
        print(f"  Unable to list outputs: {exc}")


TERMINAL_STATES = {"FINISHED", "FAILED", "CANCELLED", "STOPPED"}
FAILED_STATES = {"FAILED", "CANCELLED", "STOPPED"}


def wait_for_job(job_uuid: str, poll_seconds: int = 20, max_polls: int = 120) -> dict:
    for attempt in range(1, max_polls + 1):
        status = get_job_status(job_uuid)
        state = status.get("status") or status.get("state")
        print(f"[{attempt:03d}] {job_uuid}: {state}")
        if state in TERMINAL_STATES:
            return status
        time.sleep(poll_seconds)
    raise TimeoutError(f"Job {job_uuid} did not reach a terminal state in time")


In [7]:
def expected_container_image(case: dict) -> str:
    return f"docker://ghcr.io/wmobley/{case['image_name']}:{EXPECTED_IMAGE_TAG}"
app_catalog = {}
registration_errors = []
for app_id, case in APP_CASES.items():
    app_def = get_app(app_id)
    app_catalog[app_id] = app_def
    expected_image = expected_container_image(case)
    scheduler_args = scheduler_option_args(app_def)
    version_ok = app_def["version"] == EXPECTED_APP_VERSION
    image_ok = app_def["containerImage"] == expected_image
    allocation_ok = ALLOCATION_SCHEDULER_ARG in scheduler_args
    portal_tag_ok = EXPECTED_PORTAL_TAG in app_def.get("tags", [])

    print(f"\n{app_id}")
    print(f"  version:    {app_def['version']} ({'ok' if version_ok else 'expected ' + EXPECTED_APP_VERSION})")
    print(f"  image:      {app_def['containerImage']} ({'ok' if image_ok else 'expected ' + expected_image})")
    print(f"  portal tag: {'ok' if portal_tag_ok else 'missing ' + EXPECTED_PORTAL_TAG}")
    print(f"  allocation: {'ok' if allocation_ok else 'will submit job-level ' + ALLOCATION_SCHEDULER_ARG}")
    print(f"  enabled:    {case['enabled']}")
    print(f"  expect:     {case['expect']}")
    print("  scheduler options:")
    for arg in scheduler_args:
        print(f"    - {arg}")
    print("  file inputs:")
    for item in app_def.get("jobAttributes", {}).get("fileInputs", []):
        print(f"    - {item['name']}: {item['targetPath']}")

    if case.get("enabled", False) and not version_ok:
        registration_errors.append(f"{app_id} version is {app_def['version']}, expected {EXPECTED_APP_VERSION}")
    if case.get("enabled", False) and not image_ok:
        registration_errors.append(f"{app_id} image is {app_def['containerImage']}, expected {expected_image}")
    if case.get("enabled", False) and not portal_tag_ok:
        registration_errors.append(f"{app_id} tags are missing {EXPECTED_PORTAL_TAG!r}")
    if case.get("enabled", False):
        registered_inputs = set(get_named_file_inputs(app_def))
        missing_inputs = sorted(set(case.get("file_inputs", {})) - registered_inputs)
        if missing_inputs:
            registration_errors.append(
                f"{app_id} is missing registered file inputs: {', '.join(missing_inputs)}. "
                "Re-register the app from the updated local app.json before submitting jobs."
            )
if registration_errors:
    raise RuntimeError("Registered app definitions are not updated:\n" + "\n".join(registration_errors))



modflow-usg-simulation
  version:    0.0.f785424 (ok)
  image:      docker://ghcr.io/wmobley/modflow-usg:sha-f785424 (ok)
  portal tag: ok
  allocation: ok
  enabled:    True
  expect:     FINISHED
  scheduler options:
    - --tapis-profile tacc-apptainer
    - --job-name ${JobName}-tap_
    - -A PT2050-DataX
  file inputs:
    - mfusg-simulation-archive: simulation.zip
    - mfusg-nam: provided/model.nam
    - mfusg-bas: provided/model.bas
    - mfusg-dis: provided/model.dis
    - mfusg-drn: provided/model.drn
    - mfusg-evt: provided/model.evt
    - mfusg-ghb: provided/model.ghb
    - mfusg-gnc: provided/model.gnc
    - mfusg-hfb: provided/model.hfb
    - mfusg-lpf: provided/model.lpf
    - mfusg-oc: provided/model.oc
    - mfusg-rch: provided/model.rch
    - mfusg-riv: provided/model.riv
    - mfusg-sms: provided/model.sms
    - mfusg-wel: provided/model.wel
    - mfusg-support-01: support/support-01
    - mfusg-support-02: support/support-02
    - mfusg-support-03: support/suppor

In [8]:
def run_case(app_id: str, case: dict, wait: bool = True) -> dict:
    app_def = app_catalog[app_id]
    body = make_job_body(app_def, case)
    print("Submitting job body:")
    pprint(body)
    job = submit_job(body)
    job_uuid = job["uuid"]
    print(f"Submitted {app_id} -> {job_uuid}")
    result = {"app_id": app_id, "job_uuid": job_uuid, "submit": job}
    if wait:
        status = wait_for_job(job_uuid)
        result["status"] = status
        state = status.get("status") or status.get("state")
        if state in FAILED_STATES:
            print_job_diagnostics(job_uuid)
        try:
            result["outputs"] = list_outputs(job_uuid)
        except Exception as exc:
            result["outputs_error"] = str(exc)
    return result


In [ ]:
# Run all enabled test cases.
results = {}
for app_id, case in APP_CASES.items():
    if not case.get("enabled", False):
        print(f"Skipping {app_id}: {case['expect']}")
        continue
    print("\n" + "=" * 80)
    print(f"Running {app_id}: {case['description']}")
    results[app_id] = run_case(app_id, case, wait=True)


Running modflow-usg-simulation: Carrizo-Wilcox MODFLOW-USG generated from package inputs
Submitting job body:
{'appId': 'modflow-usg-simulation',
 'appVersion': '0.0.f785424',
 'fileInputs': [{'name': 'mfusg-bas',
                 'sourceUrl': 'tapis://ptdatax.project.PTDATAX-272/workingGAMs/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.bas'},
                {'name': 'mfusg-dis',
                 'sourceUrl': 'tapis://ptdatax.project.PTDATAX-272/workingGAMs/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.dis'},
                {'name': 'mfusg-drn',
                 'sourceUrl': 'tapis://ptdatax.project.PTDATAX-272/workingGAMs/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.drn'},
                {'name': 'mfusg-evt',
                 'sourceUrl': 'tapis://ptdatax.project.PTDATAX-272/workingGAMs/Carrizo-Wilcox-central/gmv-modflow-usg-Modified/gma12.evt'},
                {'name': 'mfusg-ghb',
                 'sourceUrl': 'tapis://ptdatax.project.PTDATAX-272/workin

In [ ]:
# Summarize final states.
summary = {}
for app_id, result in results.items():
    status = result.get("status", {})
    summary[app_id] = {
        "job_uuid": result["job_uuid"],
        "state": status.get("status") or status.get("state"),
        "expect": APP_CASES[app_id]["expect"],
    }
summary

In [ ]:
# Inspect one job in more detail, including archived output listing.
APP_ID_TO_INSPECT = "modflow6-simulation"

job_uuid = results[APP_ID_TO_INSPECT]["job_uuid"]
job_detail = get_job(job_uuid)
output_listing = list_outputs(job_uuid)

print("Job detail:")
pprint(job_detail)
print("\nArchived outputs:")
for item in output_listing:
    print(f"- {item.get('path', item.get('name'))}")

In [ ]:
# Diagnose specific failed jobs by querying job detail, status, history/messages, and archived outputs.
FAILED_JOB_UUIDS = ['a38bddd4-d316-4714-bec8-c8548f421a46-007'
    # "29dc9ea9-d5a2-4e5b-a862-b59cfe84342b-007",
    # "6451992a-c078-40e6-bd7f-7978eb88bf02-007",
    # "4e3994e9-f6a7-4531-901a-d4cedd0325a4-007",
]

for job_uuid in FAILED_JOB_UUIDS:
    print_job_diagnostics(job_uuid)
